# Experiments

## Setup: Import Libraries and Scripts

In [7]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script
import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
    {
        'name': 'Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

experiment_2 = [
        {
        'name': 'Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2 - No Bandit',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [13]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Adjusted F1 Macro': metrics['adjusted_f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2 ===
Generation 1







Evaluating population: 100%|██████████| 4/4 [00:37<00:00,  9.39s/it]


Scores: [0.8901098901098901, 0.6703296703296704, 0.6, 0.5833333333333333]
Generation 2







Evaluating population: 100%|██████████| 4/4 [00:46<00:00, 11.63s/it]


Scores: [0.696969696969697, 0.6875, 0.5238095238095238, 0.375]



To complete your explanation of the strategy application for Option 3:

*   **Option 3** retains the statutory context, which provides the legal framework and general principles for determining unfairness, but removes the specific contract context. This tests whether the model can apply general legal principles to the clause in isolation, or if it needs the specific contractual details to make a judgment.

This approach allows for a systematic evaluation of how much each piece of information (statutory context, contract context, or both) contributes to the model's ability to accurately classify the fairness of a clause. It's a well-designed experiment to understand the model's dependencies.'

Here's a continuation of your explanation for Option 3:

*   **Option 3** retains the statutory context, which provides the legal framework for determining fairness, but removes the specific contract context. This tests whether the general legal principles are sufficient for classification, or if

Generation 3







**Which option do you think is the most effective for testing the model's ability to classify the fairness of a clause?**'

Evaluating population: 100%|██████████| 4/4 [00:43<00:00, 10.83s/it]


Scores: [0.696969696969697, 0.5833333333333333, 0.5238095238095238, 0.39375]
Best Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Remove any opinionated content and focus solely on objective analysis. Respond only with '0' or '1'.
Best Template: Here's the improved template, incorporating separators and formatting for better readability:

**--- Classification Task ---**

**Instruction:**
***<instruction>***

**Clause Text:**
***<clause>***

--- Optional Context ---

**Statutory Context:**
***<statutory_context>***

**Contract Context:**
***<contract_context>***

**--- Response ---**
0 or 1

Running on test set...
Test Adjusted Macro F1: 0.5312


## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [14]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['Adjusted F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

Evaluating population:  75%|███████▌  | 3/4 [09:49<03:16, 196.58s/it]


,Experiment Name,Script,generations,pop_size,train_sample_size,test_sample_size,model_name,use_bandit_instr,use_bandit_template,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Adjusted F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),Run Time
0,Full Optimization - w META INSTRUCTION 2 & META TEMPLATE 2,optimize,3.000000,4.000000,10.000000,20,google/gemini-2.5-flash-lite-preview-06-17,True,True,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Remove any opinionated content and focus solely on objective analysis. Respond only with '0' or '1'.,"Here's the improved template, incorporating separators and formatting for better readability: **--- Classification Task ---** **Instruction:** ****** **Clause Text:** ****** --- Optional Context --- **Statutory Context:** ****** **Contract Context:** ****** **--- Response ---** 0 or 1",20,20,20,0.700000,0.142857,1.000000,0.700000,0.531250,0.531250,19.0 / 1.0,"1, 0","1, 0",precision recall f1-score support 0 1.0000 0.6842 0.8125 19 1 0.1429 1.0000 0.2500 1 accuracy 0.7000 20 macro avg 0.5714 0.8421 0.5312 20 weighted avg 0.9571 0.7000 0.7844 20,"{'0': {'precision': 1.0, 'recall': 0.6842105263157895, 'f1-score': 0.8125, 'support': 19.0}, '1': {'precision': 0.14285714285714285, 'recall': 1.0, 'f1-score': 0.25, 'support': 1.0}, 'accuracy': 0.7, 'macro avg': {'precision': 0.5714285714285714, 'recall': 0.8421052631578947, 'f1-score': 0.53125, 'support': 20.0}, 'weighted avg': {'precision': 0.9571428571428571, 'recall': 0.7, 'f1-score': 0.784375, 'support': 20.0}}",2025-07-22 14:22:15
1,zero-shot baseline,zero_shot,nan,nan,nan,100,google/gemini-2.5-flash-lite-preview-06-17,nan,nan,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Clause: Statutory Context: Contract Context:,100,100,100,0.750000,0.178571,0.714286,0.750000,0.567100,nan,93.0 / 7.0,"1, 0","1, 0",precision recall f1-score support 0 0.9722 0.7527 0.8485 93 1 0.1786 0.7143 0.2857 7 accuracy 0.7500 100 macro avg 0.5754 0.7335 0.5671 100 weighted avg 0.9167 0.7500 0.8091 100,"{'0': {'precision': 0.9722222222222222, 'recall': 0.7526881720430108, 'f1-score': 0.8484848484848485, 'support': 93.0}, '1': {'precision': 0.17857142857142858, 'recall': 0.7142857142857143, 'f1-score': 0.2857142857142857, 'support': 7.0}, 'accuracy': 0.75, 'macro avg': {'precision': 0.5753968253968254, 'recall': 0.7334869431643625, 'f1-score': 0.5670995670995671, 'support': 100.0}, 'weighted avg': {'precision': 0.9166666666666667, 'recall': 0.75, 'f1-score': 0.8090909090909091, 'support': 100.0}}",2025-07-22 13:17:31
